# 🏛️ Delentia OS v0.5 — 4-Pillars LoRA Adapter Finetuning Pipeline

> **Operating System**: **Delentia OS v0.5**  
> **Model Engine Base**: `Qwen/Qwen3.6-27B-Instruct` (Apache 2.0)  
> **Target Adapters**: 4 Asymmetric LoRA Pillars (Executor, Guardian, Router, Scribe)  
> **Output HF Repos**:
> - `Delentia/jitna-executor-v0.5`
> - `Delentia/jitna-guardian-v0.5`
> - `Delentia/jitna-router-v0.5`
> - `Delentia/jitna-scribe-v0.5`

---

## 🧬 4-Pillars Architecture Overview
| Pillar | Target Module | Hyperparameters | Purpose |
|---|---|---|---|
| **Executor** | All linear modules | r=32, alpha=64, lr=3e-5 | Deterministic TOON/JSON generation (0.00% Syntax Error) |
| **Guardian** | All linear modules | r=32, alpha=64,  lr=5e-5 | FDIA Security Veto (A=0 -> F=0.00) |
| **Router**   | All linear modules | r=32, alpha=64,  lr=5e-5 | Intent Classification & D, I, A Parameter Assignment |
| **Scribe**   | All linear modules | r=64, alpha=128, lr=5e-5 | Delta Engine 262K Context Compression (DELTA_COMPRESS) |

## 📦 Step 1: Dependencies & Environment Audit

In [ ]:
!pip install -q "datasets>=3.4.1,<4.0.0"
!pip install -q "trl>=0.19.0,<=0.24.0"
!pip install -q --no-deps bitsandbytes accelerate xformers peft triton cut_cross_entropy
!pip install -q sentencepiece protobuf huggingface_hub hf_transfer hypothesis pytest pandas pyarrow
!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps unsloth_zoo

import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
print(f"🔥 GPU: {gpu_name}")
assert "A100" in gpu_name or "L4" in gpu_name, "⚠️ Recommended A100 or L4 runtime!"

## 🔑 Step 2: Hugging Face Auth & Repository Setup

In [ ]:
import os, sys, subprocess, zipfile
from huggingface_hub import notebook_login

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
except Exception as e:
    print(f"⚠️ Drive/secrets notice: {e}")

notebook_login()

REPO_DIR = '/content/Delentia-AI-SLM'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/delentia-labs/Delentia-AI-SLM.git', REPO_DIR], check=True)

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"✅ Working directory: {os.getcwd()}")

## ⚡ Step 3: Run 4-Pillars Re-Anchoring Pipeline Script

In [ ]:
# Execute the full 4-Pillars training pipeline sequentially and push adapter weights to Hugging Face
!python training/re_anchoring_pipeline.py --all --push-to-hub

print("\n✅ All 4 Pillars re-anchored, attested, and weights uploaded to HF!")

## 🧪 Step 4: Run Three-Body Synthesis Verification

In [ ]:
# Run consensus & variance stress test across 3 active slots
!python training/test_three_body_synthesis.py --gguf-path delentia-slm-jitna-v0.5-27B.gguf

print("\n🎉 4-Pillars Verification Complete!")

## 📝 Step 5: Generate and Upload Model Cards (README.md)

In [ ]:
# Upload detailed Model Cards for all 4 Pillars to Hugging Face
!python upload_4_pillars_model_cards.py

print("\n🎉 Deployment Complete! Both Weights and Model Cards are live on Hugging Face.")